# Hypothesis testing pattern

Scientific reasoning follows a systematic process: observe phenomena, form hypotheses to explain them, test those hypotheses against evidence, and refine understanding based on results. This methodical approach is powerful because it explicitly separates speculation from verification, allows multiple competing explanations to coexist, and iteratively improves accuracy through structured feedback.

The Hypothesis Testing pattern brings this scientific method to AI reasoning. Instead of directly answering a question, the system generates several plausible hypotheses, evaluates each independently against established knowledge, scores their likelihood, and synthesizes the best-supported explanation into a final conclusion. This approach is particularly valuable when multiple explanations are genuinely possible, when evidence is incomplete or ambiguous, or when a systematic audit of alternatives improves on an intuitive single-shot answer.

In this notebook we implement hypothesis testing step by step - one function at a time - building toward a complete pipeline that formulates multiple hypotheses, scores each one, and arrives at a well-supported conclusion.

In [1]:
import os
from collections import namedtuple
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

### Initialize the language model

In [2]:
# temperature=0.7 introduces enough variation that repeated generation calls
# produce genuinely different hypotheses rather than paraphrases of each other
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

## Representing a hypothesis
Each hypothesis needs to carry four pieces of information: an identifier so we can refer to it by number, the actual claim it makes, the numeric score assigned after evaluation, and the model's written reasoning behind that score.

A `namedtuple` is the right tool here. It is immutable — once a hypothesis has been scored we never modify it in place — and it provides attribute-style access (`h.score`, `h.statement`) without requiring a full class definition with `__init__` and `__repr__`. Keeping it simple also makes the subsequent sorting and indexing code easy to read.

In [3]:
# Four fields — every field is populated before a Hypothesis is ever stored
Hypothesis = namedtuple('Hypothesis', ['id', 'statement', 'score', 'reasoning'])
# id        — integer label (1, 2, 3 …) used in progress output and display
# statement — the plain-text claim: "X happens because Y"
# score     — float 0–10 returned by score_hypothesis(); drives ranking
# reasoning — one or two sentences the model used to justify its score;
#             passed into the conclusion prompt so synthesis has rich material

Because `Hypothesis` is a `namedtuple`, a list of them can be ranked with `sorted(..., key=lambda h: h.score, reverse=True)` and the best hypothesis is simply `hypotheses[0]` - both operations appear inside `test_hypotheses()` below. Immutability also means that once `score_hypothesis()` produces a result, nothing downstream can accidentally overwrite the score after the fact.

## Generating candidate hypotheses
The first step of hypothesis testing is brainstorming: we ask the model to produce several distinct, plausible explanations for the question without yet judging any of them. Generating multiple hypotheses before scoring any of them ensures that evaluation pressure does not collapse diversity - a single-shot question often collapses to the model's most probable prior answer, whereas asking for *N* distinct alternatives forces it to explore the explanation space before committing.

In [4]:
def generate_hypotheses(question: str, num_hypotheses: int = 3) -> List[str]:
    """Return num_hypotheses plain-text hypothesis statements for the question."""
    # Ask for exactly num_hypotheses numbered statements; no extra commentary
    prompt = (
        f"Generate exactly {num_hypotheses} distinct, plausible hypotheses that could "
        f"explain or answer the following question.\n\n"
        f"Question: {question}\n\n"
        f"Format each hypothesis as a single sentence prefixed with its number and a "
        f"period (e.g. '1. The sky is blue because ...'). "
        f"Output only the numbered lines, nothing else."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    statements = []
    for line in raw.splitlines():
        line = line.strip()
        # A valid hypothesis line starts with a digit followed by a period
        if line and line[0].isdigit() and '.' in line:
            _, _, text = line.partition('.')   # discard the "N." prefix
            text = text.strip()
            if text:
                statements.append(text)

    # Fallback: if the model ignored the formatting instruction, take any non-empty line
    if not statements:
        statements = [l.strip() for l in raw.splitlines() if l.strip()]

    return statements[:num_hypotheses]   # cap at the requested count in case of over-generation

The function asks the model for numbered lines and parses each one by splitting on the first period to discard the `"N."` prefix. The fallback path (any non-empty line) exists for models that ignore the formatting instruction - in practice `gpt-4o-mini` follows it reliably, so the fallback rarely fires. Capping with `[:num_hypotheses]` prevents the occasional over-generation from producing more candidates than requested.

In [5]:
# Quick test: confirm we get back the right number of statements
test_q = "Why do leaves change color in autumn?"
test_statements = generate_hypotheses(test_q, num_hypotheses=3)

print(f"Question: {test_q}\n")
for i, s in enumerate(test_statements, 1):
    print(f"  {i}. {s}")

Question: Why do leaves change color in autumn?

  1. Leaves change color in autumn due to the breakdown of chlorophyll, revealing other pigments such as carotenoids and anthocyanins that were previously masked.
  2. The decrease in daylight and cooler temperatures during autumn trigger biochemical changes in leaves, leading to the synthesis of new pigments that create vibrant colors.
  3. Environmental stressors, such as drought or nutrient deficiency, during the growing season can enhance the production of pigments like anthocyanins, resulting in more intense leaf colors in autumn.


## Scoring a single hypothesis
Once we have the candidate statements we evaluate each one independently. The model assigns a score on a 0–10 scale and explains its reasoning in one or two sentences. Asking for both a number and a written justification is important: the number drives ranking, but the reasoning text becomes part of the final conclusion prompt, giving the synthesis step richer material than a bare list of scores would allow.

In [6]:
def score_hypothesis(question: str, hypothesis: str) -> tuple:
    """Ask the LLM to score one hypothesis 0-10 and state its reasoning.

    Returns (score: float, reasoning: str).
    """
    prompt = (
        f"You are evaluating a hypothesis for the following question.\n\n"
        f"Question: {question}\n\n"
        f"Hypothesis: {hypothesis}\n\n"
        f"Score this hypothesis from 0 to 10 using three criteria:\n"
        f"  - Plausibility: how likely it is to be true given common knowledge\n"
        f"  - Evidence alignment: how well it matches established facts\n"
        f"  - Specificity: whether it makes a concrete, testable claim\n\n"
        f"Respond in exactly this format (two lines, nothing else):\n"
        f"SCORE: <number from 0 to 10>\n"
        f"REASONING: <one or two sentences explaining the score>"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    score = 5.0       # neutral fallback if we fail to parse a number
    reasoning = raw   # fallback: treat the whole response as the reasoning

    for line in raw.splitlines():
        line = line.strip()
        if line.upper().startswith('SCORE:'):
            score_text = line.split(':', 1)[1].strip()   # everything after "SCORE:"
            try:
                score = float(score_text)
                score = max(0.0, min(10.0, score))        # clamp to valid range [0, 10]
            except ValueError:
                pass                                      # leave score as 5.0 fallback
        elif line.upper().startswith('REASONING:'):
            reasoning = line.split(':', 1)[1].strip()    # everything after "REASONING:"

    return score, reasoning

Three-criterion scoring gives the model a rubric that goes beyond "does this feel right." Plausibility catches wild guesses; evidence alignment penalises claims that contradict established knowledge; specificity rewards hypotheses that make falsifiable predictions over vague hand-waving. Clamping the parsed float to `[0, 10]` prevents an off-range response like `"10.5"` from distorting the downstream ranking.

In [7]:
# Quick test: score one of the statements generated above
test_score, test_reasoning = score_hypothesis(test_q, test_statements[0])

print(f"Hypothesis: {test_statements[0]}\n")
print(f"Score:     {test_score:.1f} / 10")
print(f"Reasoning: {test_reasoning}")

Hypothesis: Leaves change color in autumn due to the breakdown of chlorophyll, revealing other pigments such as carotenoids and anthocyanins that were previously masked.

Score:     9.0 / 10
Reasoning: The hypothesis is highly plausible and aligns well with established botanical knowledge regarding chlorophyll breakdown and the visibility of other pigments, making it a solid and testable claim.


## Testing all hypotheses together
With generation and scoring working independently, we can wire them together. `test_hypotheses` takes the raw statements from `generate_hypotheses`, scores each one, wraps the results in `Hypothesis` namedtuples, and returns a list sorted from highest score to lowest. Sorting here means that everything downstream - `conclude`, the display loop, the "best hypothesis" reference — simply reads `hypotheses[0]` instead of calling `max` repeatedly.

In [8]:
def test_hypotheses(question: str, statements: List[str]) -> List[Hypothesis]:
    """Score every statement and return Hypothesis namedtuples ranked best-first."""
    hypotheses = []
    for i, statement in enumerate(statements, start=1):
        print(f"  Scoring hypothesis {i}/{len(statements)}...")   # progress indicator
        score, reasoning = score_hypothesis(question, statement)
        # Assemble the immutable record — score and reasoning never change after this
        hypotheses.append(Hypothesis(id=i, statement=statement, score=score, reasoning=reasoning))

    # Sort descending so index 0 is always the strongest hypothesis
    return sorted(hypotheses, key=lambda h: h.score, reverse=True)

Scoring is intentionally kept independent per hypothesis: each call to `score_hypothesis` sees only the question and one statement, not the other candidates. This prevents the model from anchoring its scores on whichever hypothesis happens to appear first - each score reflects only that hypothesis's own merits against the evaluation rubric.

## Synthesising a conclusion
Selecting the highest-scoring hypothesis is a reasonable answer, but a better one acknowledges the alternatives. `conclude` passes the full ranked list - scores *and* reasoning — to the model and asks it to write a nuanced conclusion. The model can then say "the primary explanation is X (8.2/10), though Y (6.5/10) plays a secondary role" in a way that a simple argmax cannot.

In [9]:
def conclude(question: str, hypotheses: List[Hypothesis]) -> str:
    """Synthesise a final answer from the full ranked list of hypotheses.

    Rather than picking the top scorer and discarding the rest, the LLM uses
    the complete ranking — scores and reasoning — to write a nuanced conclusion
    that can acknowledge complementary or competing explanations.
    """
    # Build a compact ranked summary: "[score] statement — reasoning"
    ranked_summary = "\n".join(
        f"  {i+1}. [{h.score:.1f}/10] {h.statement}  — {h.reasoning}"
        for i, h in enumerate(hypotheses)   # hypotheses are already sorted best-first
    )
    prompt = (
        f"The following hypotheses were generated and scored for the question below.\n\n"
        f"Question: {question}\n\n"
        f"Ranked hypotheses (highest score first):\n{ranked_summary}\n\n"
        f"Write a final conclusion that:\n"
        f"  - Leads with the most strongly supported explanation\n"
        f"  - Briefly acknowledges credible alternative explanations\n"
        f"  - Notes any remaining uncertainty\n\n"
        f"Keep the conclusion to 3-4 sentences."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()   # plain string — callers decide how to display it

Providing the reasoning strings - not just the numeric scores - gives the LLM enough context to write a substantive conclusion. A prompt that showed only `[8.2, 6.5, 4.1]` would produce generic hedging; providing the actual justifications lets the synthesis reference specific evidence from the evaluation phase and produce a conclusion that reads as a coherent argument rather than a hedge.

## The full hypothesis-testing pipeline
With all four building blocks in place - `generate_hypotheses`, `score_hypothesis`, `test_hypotheses`, and `conclude` - the driver function simply chains them together and prints progress at each step so we can watch the reasoning unfold. The driver owns no state of its own; every intermediate result passes through as a return value.

In [10]:
def hypothesis_test(question: str, num_hypotheses: int = 3) -> dict:
    """Full pipeline: generate → score → conclude.

    Returns a dict with all intermediate data so callers can inspect any step.
    """
    print(f"Question: {question}\n")

    # Step 1 — Generate candidate statements without yet scoring them
    print(f"Step 1 — Generating {num_hypotheses} hypotheses...")
    statements = generate_hypotheses(question, num_hypotheses)
    for i, s in enumerate(statements, 1):
        print(f"  H{i}: {s}")

    # Step 2 — Score each hypothesis independently, then rank highest-first
    print(f"\nStep 2 — Scoring hypotheses...")
    hypotheses = test_hypotheses(question, statements)

    # Step 3 — Synthesise a nuanced conclusion from the full ranked list
    print(f"\nStep 3 — Synthesising conclusion...")
    conclusion = conclude(question, hypotheses)

    return {
        "question":   question,
        "hypotheses": hypotheses,   # List[Hypothesis], already sorted best-first
        "conclusion": conclusion,
    }

Returning a plain dict keeps `hypothesis_test` testable without depending on any class structure: a caller can assert `len(result["hypotheses"]) == num_hypotheses` or check `result["hypotheses"][0].score > 5` without importing a custom class. Each key maps directly to data that was already computed, so the dict adds no duplication and imposes no hidden state.

## Solving a problem with hypothesis testing
Let's run the full pipeline on a question that genuinely benefits from considering multiple explanations: why overfitting happens in machine learning. A direct single-call answer typically gives the single most common explanation (too many parameters, not enough data); hypothesis testing forces the model to articulate the full landscape of causes before committing to any of them.

In [11]:
question = "Why might a machine learning model perform well on training data but poorly on new data?"

result = hypothesis_test(question, num_hypotheses=3)

print("\n" + "=" * 65)
print("RESULTS")
print("=" * 65)

print("\nHypotheses (ranked best → worst):\n")
for h in result["hypotheses"]:
    print(f"  [{h.score:.1f}/10]  H{h.id}: {h.statement}")
    print(f"               {h.reasoning}\n")

print("Conclusion:\n")
print(result["conclusion"])

Question: Why might a machine learning model perform well on training data but poorly on new data?

Step 1 — Generating 3 hypotheses...
  H1: The model may have overfitted to the training data by capturing noise and specific patterns that do not generalize to new data.
  H2: The training data may not be representative of the new data, leading to discrepancies in feature distributions that the model cannot adapt to.
  H3: The model could be too complex relative to the amount of training data, resulting in a high variance that affects its performance on unseen examples.

Step 2 — Scoring hypotheses...
  Scoring hypothesis 1/3...
  Scoring hypothesis 2/3...
  Scoring hypothesis 3/3...

Step 3 — Synthesising conclusion...

RESULTS

Hypotheses (ranked best → worst):

  [9.0/10]  H1: The model may have overfitted to the training data by capturing noise and specific patterns that do not generalize to new data.
               The hypothesis is highly plausible and aligns well with established 

Notice that the ranked list often reveals a nuanced picture: two or three hypotheses may score closely (for example 7–8 out of 10) because they are complementary rather than competing explanations. The conclusion step captures this by weaving them into a single coherent answer, rather than treating hypothesis testing as a winner-take-all race where every alternative is simply discarded.

## Comparing hypothesis testing with direct answering
To make the benefit concrete, we can run both approaches on the same question and compare. Direct answering is a single LLM call; hypothesis testing is `num_hypotheses` scoring calls plus one synthesis call. The extra cost buys explicit exploration of alternatives and a scored justification for the final answer - which also makes the reasoning auditable.

In [12]:
def direct_answer(question: str) -> str:
    """Baseline: one LLM call, no intermediate reasoning structure."""
    response = llm.invoke([HumanMessage(content=f"Answer briefly: {question}")])
    return response.content.strip()


comp_question = "Why does the Moon appear larger near the horizon than high in the sky?"

print(f"Question: {comp_question}\n")

# Baseline — a single, unstructured call
print("Direct answer (1 LLM call):")
print(direct_answer(comp_question))

# Hypothesis testing — generate, score independently, then synthesise
print("\n" + "-" * 60)
print("Hypothesis testing (num_hypotheses score calls + 1 synthesis call):\n")
comp_result = hypothesis_test(comp_question, num_hypotheses=3)

print("\nFinal conclusion:")
print(comp_result["conclusion"])

Question: Why does the Moon appear larger near the horizon than high in the sky?

Direct answer (1 LLM call):
The Moon appears larger near the horizon due to an optical illusion known as the "Ponzo illusion." When the Moon is near the horizon, our brain compares it to objects on the ground, such as trees and buildings, making it seem larger. In contrast, when it's high in the sky, there are fewer reference points, and it appears smaller.

------------------------------------------------------------
Hypothesis testing (num_hypotheses score calls + 1 synthesis call):

Question: Why does the Moon appear larger near the horizon than high in the sky?

Step 1 — Generating 3 hypotheses...
  H1: The Moon appears larger near the horizon due to an optical illusion known as the Ponzo illusion, where our brains perceive the Moon's size in relation to objects on the horizon.
  H2: Atmospheric effects, such as increased air density and scattering near the horizon, may cause the Moon to appear larger

**When to use hypothesis testing:** Diagnostic reasoning where multiple causes are genuinely plausible - system failures, performance regressions, root-cause analysis. Scientific or analytical questions where the literature supports competing theories. Situations where acknowledging uncertainty is as important as giving a single answer.

**Limitations to be aware of:** The quality of the ranked list depends on diversity in the generation step - if `generate_hypotheses` produces near-identical statements, scoring cannot discriminate meaningfully. Each run involves `num_hypotheses + 1` LLM calls (one per score, plus one synthesis), which costs proportionally more than a direct single-call answer. For questions with a single, well-established answer this overhead buys nothing.

**Extending the pattern:** Replace `score_hypothesis` with a retrieval-augmented version that fetches supporting documents before scoring. Add a second generation round that proposes *counter-hypotheses* to the highest scorers. Use iterative refinement - feed the conclusion back as the question for a second pass to sharpen the answer progressively.